In [1]:
import polars as pl
import time
import os

# Caminho absoluto do arquivo de origem
caminho_origem = r"C:\Users\Guilherme\OneDrive - Agência Nacional do Cinema\Documentos - CEM\Backup Repositório\DADOS_SCB\AG_SCB_LIST_SALA_OBRA_SESS_RES.csv" 

try:
    # Tenta ler com separador de ponto e vírgula (padrão comum no Brasil)
    lf = pl.scan_csv(caminho_origem, separator=';', infer_schema_length=10000)
    
    # A CORREÇÃO ESTÁ AQUI:
    # Usamos collect_schema().names() para forçar a leitura apenas dos cabeçalhos
    colunas = lf.collect_schema().names()

except:
    # Fallback: Se der erro, tenta com vírgula
    print("Separador ';' falhou ou arquivo não encontrado. Tentando com ','...")
    lf = pl.scan_csv(caminho_origem, separator=',', infer_schema_length=10000)
    colunas = lf.collect_schema().names()

print(f"Total de colunas encontradas: {len(colunas)}")
print("-" * 30)
print("Colunas disponíveis:")
for i, col in enumerate(colunas):
    print(f"{i}: {col}")

Total de colunas encontradas: 18
------------------------------
Colunas disponíveis:
0: ID_SESSAO_CINEMATOGRAFICA
1: ANO_CINEMATOGRAFICO
2: SEMANA_CINEMATOGRAFICA
3: PRIMEIRO_DIA_SEMANA
4: ULTIMO_DIA_SEMANA
5: DATA_SESSAO
6: DIA_SEMANA
7: QUALIFICADOR_DIA_SEMANA
8: DATA_HORA_SESSAO
9: TIPO_SESSAO
10: REGISTRO_SALA
11: REGISTRO_COMPLEXO
12: REGISTRO_EXIBIDOR
13: CPB_ROE
14: TITULO
15: NACIONALIDADE
16: PUBLICO
17: RENDA


In [2]:

# 1. Definição das colunas
cols_to_keep = [
    "ID_SESSAO_CINEMATOGRAFICA", # 0
    "DATA_HORA_SESSAO",          # 8
    "TIPO_SESSAO",               # 9
    "REGISTRO_SALA",             # 10
    "CPB_ROE"                    # 13
]

caminho_origem = r"C:\Users\Guilherme\OneDrive - Agência Nacional do Cinema\Documentos - CEM\Backup Repositório\DADOS_SCB\AG_SCB_LIST_SALA_OBRA_SESS_RES.csv"
nome_arquivo_saida = "AG_SCB_SESSAO_REDUZIDO.csv"

print("Iniciando leitura Eager com encoding Latin-1...")
start_time = time.time()

try:
    # 2. MUDANÇA AQUI: Usamos read_csv (Eager) em vez de scan_csv (Lazy)
    # O parâmetro 'columns' garante que só carregamos o necessário, economizando RAM/CPU.
    df_reduzido = pl.read_csv(
        caminho_origem, 
        separator=';', 
        encoding='latin-1', # Agora funciona!
        columns=cols_to_keep, # Faz o filtro durante a leitura
        ignore_errors=True    # Segurança extra caso haja alguma linha corrompida
    )
    
    tempo_leitura = time.time() - start_time
    print(f"Leitura concluída em: {tempo_leitura:.2f} segundos")
    print(f"Shape do dataset: {df_reduzido.shape}")

    # 3. Salva o novo arquivo CSV reduzido
    # O Polars salvará em UTF-8 por padrão, resolvendo problemas futuros
    df_reduzido.write_csv(nome_arquivo_saida, separator=';')
    print(f"Arquivo '{nome_arquivo_saida}' salvo com sucesso em: {os.getcwd()}")

    # 4. Mostra uma amostra
    print("-" * 30)
    print(df_reduzido.head())

except Exception as e:
    print(f"Erro: {e}")

Iniciando leitura Eager com encoding Latin-1...
Leitura concluída em: 5.66 segundos
Shape do dataset: (12137944, 5)
Arquivo 'AG_SCB_SESSAO_REDUZIDO.csv' salvo com sucesso em: C:\Users\Guilherme\OneDrive\Cursos\ENAP\MBA em Inteligência Artificial\Disciplinas\21 - Machine Learning\Trabalho Final\Notebooks
------------------------------
shape: (5, 5)
┌──────────────────┬──────────────────┬────────────────┬───────────────┬────────────────┐
│ ID_SESSAO_CINEMA ┆ DATA_HORA_SESSAO ┆ TIPO_SESSAO    ┆ REGISTRO_SALA ┆ CPB_ROE        │
│ TOGRAFICA        ┆ ---              ┆ ---            ┆ ---           ┆ ---            │
│ ---              ┆ str              ┆ str            ┆ i64           ┆ str            │
│ i64              ┆                  ┆                ┆               ┆                │
╞══════════════════╪══════════════════╪════════════════╪═══════════════╪════════════════╡
│ 58160528         ┆ 2023-01-29T00:35 ┆ Sessão Regular ┆ 5004747       ┆ E2200401500000 │
│                  ┆

In [3]:


# Pega o dataframe que já está na memória (df_reduzido)
print("Salvando em Parquet...")
start_time = time.time()

# Salva em Parquet (usa compressão 'zstd' ou 'snappy' por padrão)
nome_arquivo_parquet = "AG_SCB_SESSAO_REDUZIDO.parquet"
df_reduzido.write_parquet(nome_arquivo_parquet)

tempo_escrita = time.time() - start_time
print(f"Escrita concluída em: {tempo_escrita:.2f} segundos")

# Comparação de tamanho
tamanho_csv = os.path.getsize(nome_arquivo_saida) / (1024 * 1024 * 1024) # em GB
tamanho_parquet = os.path.getsize(nome_arquivo_parquet) / (1024 * 1024 * 1024) # em GB

print("-" * 30)
print(f"Tamanho CSV:     {tamanho_csv:.2f} GB")
print(f"Tamanho Parquet: {tamanho_parquet:.2f} GB")
print(f"Redução de espaço: {(1 - tamanho_parquet/tamanho_csv)*100:.1f}%")

Salvando em Parquet...
Escrita concluída em: 1.21 segundos
------------------------------
Tamanho CSV:     0.78 GB
Tamanho Parquet: 0.11 GB
Redução de espaço: 85.3%


In [4]:
print(pl.read_parquet("AG_SCB_SESSAO_REDUZIDO.parquet")["TIPO_SESSAO"].value_counts(sort=True))

shape: (4, 2)
┌────────────────────┬──────────┐
│ TIPO_SESSAO        ┆ count    │
│ ---                ┆ ---      │
│ str                ┆ u32      │
╞════════════════════╪══════════╡
│ Sessão Regular     ┆ 12052343 │
│ Sessão Privada     ┆ 43994    │
│ Pré-Estreia        ┆ 38906    │
│ Mostra ou Festival ┆ 2701     │
└────────────────────┴──────────┘


In [5]:
print(pl.read_parquet("AG_SCB_SESSAO_REDUZIDO.parquet")["TIPO_SESSAO"].null_count())

0


In [6]:


df = pl.read_parquet("AG_SCB_SESSAO_REDUZIDO.parquet", columns=["DATA_HORA_SESSAO"])
print("Min (txt):", df["DATA_HORA_SESSAO"].min())
print("Max (txt):", df["DATA_HORA_SESSAO"].max())
print("\nAmostra:", df["DATA_HORA_SESSAO"].head(3).to_list())

Min (txt): 2023-01-05T07:20:00Z
Max (txt): 2025-11-15T01:10:00Z

Amostra: ['2023-01-29T00:35:00Z', '2023-02-03T21:30:00Z', '2023-02-03T22:00:00Z']


In [7]:
import polars as pl

caminho_origem = r"C:\Users\Guilherme\OneDrive - Agência Nacional do Cinema\Documentos - CEM\Backup Repositório\DADOS_SCB\AG_SCB_LIST_SALA_OBRA_SESS_RES.csv"

print("Lendo coluna DATA_HORA_SESSAO do CSV original...")

try:
    # Lê apenas a coluna de data (índice 8 ou nome)
    # try_parse_dates=True tenta converter para data real automaticamente para garantir a ordem correta
    df_datas = pl.read_csv(
        caminho_origem, 
        separator=';', 
        encoding='latin-1',
        columns=["DATA_HORA_SESSAO"],
        try_parse_dates=True 
    )

    print("-" * 30)
    print(f"Total de linhas: {df_datas.height}")
    print("Data Mínima:", df_datas["DATA_HORA_SESSAO"].min())
    print("Data Máxima:", df_datas["DATA_HORA_SESSAO"].max())

except Exception as e:
    print(f"Erro: {e}")

Lendo coluna DATA_HORA_SESSAO do CSV original...
------------------------------
Total de linhas: 12137944
Data Mínima: 2023-01-05 07:20:00+00:00
Data Máxima: 2025-11-15 01:10:00+00:00
